Intermediate Deep Learning - Deep Reinforcement Learning
# Project: Deep Reinforcement Learning in Trading Environment

## Overview
In this project, you will implement a deep reinforcement learning (DRL) algorithm covered in this course and train an agent in a trading environment. You will then use the observed results to guide and assess improvements to your initial implementation.

### Project Requirements

1. **Implement a course algorithm.** Choose one DRL algorithm covered in the course, implement it, and train an initial agent to establish a baseline. Explain why this algorithm is suitable for the task.
2. **Analyse the initial results.** Examine training behaviour and agent performance to identify limitations and opportunities for improvement.
3. **Optimise based on evidence.** Use your observations to motivate changes, for example to hyperparameters, network architecture, input features, reward design, or the training procedure. Compare the results with your baseline and explain what worked, what did not, and why.

### Deliverables and Schedule

- **Upload your code:** Jupyter Notebooks or Python scripts containing the algorithm implementation, training, optimisation experiments, and evaluation.
- **Upload `evaluation_results.csv`:** Generate this file for your final agent using the provided evaluation function.
- **Give a 10–15-minute presentation, followed by Q&A:** Explain your choice of algorithm, the initial results, your optimisation ideas and their rationale, the experiments conducted, and the resulting performance compared with the baseline.

No written report is required. Upload the code and evaluation CSV **before your presentation**. Presentations will take place during **the last class session**.

### Environment
We will use Gym Trading Env as our trading environment. (https://gym-trading-env.readthedocs.io/en/latest/)

- This is a gymnasium-compatible environment designed to simulate trading (stocks or crypto) from historical market data.
- Its goal is to provide a fast and customizable platform for training RL agents in a trading scenario.

We will use BTC/USDT hour step historical data from Binance for training and evaluation. All students must use the same fixed evaluation period: **2026-08-17 00:00 UTC to 2026-09-17 00:00 UTC (exclusive)**, covering all complete days from August 17 through September 16. Do not change the evaluation dates for submission. Training uses the preceding year, with no overlap.

Following code blocks demonstrate how to set up the environment and evaluate your agent.

### Grading Criteria

- **Implementation — 25%:** Correct implementation of a DRL algorithm covered in the course, with a functioning training and evaluation process.
- **Optimisation — 50%:** Analysis of observed results, well-motivated improvements, systematic experiments, and clear comparisons with the initial baseline. Explain both successful and unsuccessful changes.
- **Presentation — 25%:** Clear explanation of the algorithm choice, optimisation rationale, experimental results, and conclusions.

**Bonus:** The top-performing agent receives a **15% bonus**, based on the submitted evaluation results.


---

## Environment Setup

### Install required packages
Use a Python 3.12 Jupyter kernel with the dependencies in `requirements.txt`. The local `.venv-rl` environment is used for verification. On Intel macOS, PyTorch 2.2.2 requires NumPy <2. Internet access is required for the first market-data download. Run the commented installation command once if needed, then restart the kernel.


In [1]:
# Install once into the selected kernel; do not mutate the environment on every Run All.
# %pip install -r requirements.txt
import sys
print(sys.executable)


/Users/alivakof/Documents/Codex/python-data-science/.venv/bin/python


In [3]:
import numpy as np
import pandas as pd
import gymnasium as gym
import gym_trading_env
import json
import ssl
import certifi
from urllib.request import urlopen
from urllib.parse import urlencode
from pathlib import Path
import matplotlib.pyplot as plt
import time



### Prepare Data Set

- Create a data folder to store historical data
- Download historical data for BTC/USDT from Binance using the public Binance market-data API (no API key required).
- Preprocess the data to create features. The features (plus two dynamic features: last position taken by the agent, and the current real position) are the state of the environment at each time step.  
    *(You can add more features if you want to experiment with different state representations.)*
- Select training and evaluation data based on the specified date ranges.  
    *(You can modify the training range if you want to experiment with different time periods, however, keep in mind the evaluation period should always be after the training period.)*

All timestamps are UTC. Ranges are start-inclusive and end-exclusive. Download seven extra days before training for feature warm-up. Validate every expected hourly candle before using or caching the data; missing data raises an error instead of silently shortening evaluation. A dated cache keeps older datasets intact.

Source: [Binance Kline API](https://developers.binance.com/docs/binance-spot-api-docs/rest-api/market-data-endpoints#klinecandlestick-data).


In [ ]:
# Support running from either the Project folder or the repository root.
project_folder = Path("Project") if Path("Project/Project_Instruction.ipynb").is_file() else Path(".")
data_folder = project_folder / "data"
data_folder.mkdir(parents=True, exist_ok=True)
eval_folder = project_folder / "eval"
eval_folder.mkdir(parents=True, exist_ok=True)

# Fixed evaluation window shared by all students; end is exclusive.
EVAL_START = pd.Timestamp("2026-08-17", tz="UTC")
EVAL_END = pd.Timestamp("2026-09-17", tz="UTC")
TRAIN_START = EVAL_START - pd.DateOffset(years=1)
DOWNLOAD_START = TRAIN_START - pd.Timedelta(days=7)
data_path = data_folder / f"binance-BTCUSDT-1h-{DOWNLOAD_START:%Y%m%d}-{EVAL_END:%Y%m%d}.pkl"


def validate_hourly(frame, start, end, label):
    expected = pd.date_range(start, end, freq="h", inclusive="left")
    missing = expected.difference(frame.index)
    unexpected = frame.index.difference(expected)
    if (frame.empty or frame.index.has_duplicates or not frame.index.is_monotonic_increasing
            or len(missing) or len(unexpected)):
        raise ValueError(f"{label}: incomplete/invalid hourly data; "
                         f"missing={len(missing)}, unexpected={len(unexpected)}, "
                         f"first missing={missing[:5].tolist()}")
    values = frame[["open", "high", "low", "close", "volume"]]
    if not np.isfinite(values.to_numpy()).all():
        raise ValueError(f"{label}: non-finite OHLCV values")
    if ((values[["open", "high", "low", "close"]] <= 0).any().any()
            or (values["volume"] < 0).any()
            or (values["high"] < values[["open", "low", "close"]].max(axis=1)).any()
            or (values["low"] > values[["open", "high", "close"]].min(axis=1)).any()):
        raise ValueError(f"{label}: invalid OHLCV values")
    print(f"{label}: {len(frame):,} candles, {frame.index.min()} to {frame.index.max()}; no missing hours")


if data_path.exists():
    df = pd.read_pickle(data_path)
else:
    rows = []
    cursor = int(DOWNLOAD_START.timestamp() * 1000)
    end_ms = int(EVAL_END.timestamp() * 1000)
    while cursor < end_ms:
        query = urlencode(dict(symbol="BTCUSDT", interval="1h", startTime=cursor,
                               endTime=end_ms - 1, limit=1000))
        with urlopen("https://data-api.binance.vision/api/v3/klines?" + query, timeout=30,
                     context=ssl.create_default_context(cafile=certifi.where())) as response:
            batch = json.load(response)
        if not isinstance(batch, list) or not batch:
            raise RuntimeError(f"Binance returned no usable data at {cursor}: {batch}")
        next_cursor = int(batch[-1][0]) + 3_600_000
        if next_cursor <= cursor:
            raise RuntimeError("Binance pagination did not advance")
        rows.extend(batch)
        cursor = next_cursor
        time.sleep(0.1)
    df = pd.DataFrame([row[1:6] for row in rows],
                      columns=["open", "high", "low", "close", "volume"], dtype=float,
                      index=pd.to_datetime([row[0] for row in rows], unit="ms", utc=True))
    df.index.name = "date_open"
    df["date_close"] = pd.to_datetime([row[6] for row in rows], unit="ms", utc=True)

validate_hourly(df, DOWNLOAD_START, EVAL_END, "Downloaded data")
df.to_pickle(data_path)

# Causal features: all rolling windows use only present/past candles.
df["feature_close"] = df["close"].pct_change()
df["feature_open"] = df["open"] / df["close"]
df["feature_high"] = df["high"] / df["close"]
df["feature_low"] = df["low"] / df["close"]
df["feature_volume"] = df["volume"] / df["volume"].rolling(7 * 24).max()
df = df.replace([np.inf, -np.inf], np.nan).dropna()

df_train = df.loc[(df.index >= TRAIN_START) & (df.index < EVAL_START)].copy()
df_eval = df.loc[(df.index >= EVAL_START) & (df.index < EVAL_END)].copy()
validate_hourly(df_train, TRAIN_START, EVAL_START, "Training")
validate_hourly(df_eval, EVAL_START, EVAL_END, "Evaluation")
assert df_train.index.max() < df_eval.index.min(), "Train/eval overlap"
print(f"Validated data saved to {data_path}")


In [ ]:
df_train

In [ ]:
df_eval

### Setting up the Trading Environment

We use the `df_train` DataFrame for training and `df_eval` DataFrame for evaluation.

The `positions` parameter defines the discrete positions the agent can take, it is a list containing possible position values. A position value corresponds to the ratio of the portfolio valuation engaged in the position ( > 0 to bet on the rise, < 0 to bet on the decrease)

- if `position < 0` : the agent is shorting the asset
- if `position = 0` : the agent is out of the market
- if `position > 0` : the agent is longing the asset
- if `position = 1` : the agent is fully invested in the asset
- if `position > 1` : the agent is using leverage to invest more than its portfolio valuation in the asset

You are free to modify the `positions` list to experiment with different position options for the agent.


In [ ]:
def prepare_env_data(frame):
    # TradingEnv stores history dates without timezone information.
    # Keep UTC clock times while matching that format for save_for_render().
    frame = frame.copy()
    frame.index = pd.to_datetime(frame.index, utc=True).tz_localize(None)
    if "date_close" in frame.columns:
        frame["date_close"] = pd.to_datetime(frame["date_close"], utc=True).dt.tz_localize(None)
    return frame


POSITIONS = [ -1, 0, 0.5, 1, 2] # -1 (=SHORT), 0(=OUT), +1 (=LONG)

env_train = gym.make("TradingEnv",
        name= "BTCUSD",
        initial_position=0, verbose=0,
        df = prepare_env_data(df_train), # Your dataset with your custom features
        positions = POSITIONS,
        trading_fees = 0.01/100, # 0.01% per stock buy / sell (Binance fees)
        borrow_interest_rate= 0.0003/100, # 0.0003% per timestep (one timestep = 1h here)
    )

env_eval = gym.make("TradingEnv",
        name= "BTCUSD",
        initial_position=0, verbose=0,
        df = prepare_env_data(df_eval), # Your dataset with your custom features
        positions = POSITIONS,
        trading_fees = 0.01/100, # 0.01% per stock buy / sell (Binance fees)
        borrow_interest_rate= 0.0003/100, # 0.0003% per timestep (one timestep = 1h here)
    )


### Example of interacting with the Environment

The interaction with the environment follows the standard Gymnasium API. At each time step, the agent selects an action (position index) based on the current observation, and the environment returns the next observation, reward, and done flag.

The following code block demonstrates a simple interaction loop where the agent randomly selects actions.

The episode is terminated when `done` or `truncated` is `True`:

- When environment reaches the end of the dataset, `truncated` is set to `True`.
- When agent's portfolio valuation drops below 0, `done` is set to `True`. (This means the agent has gone bankrupt, the situdation can happen when high leverage is used.)

The reward at each time step is calculated based on the change in portfolio valuation, taking into account trading fees and borrow interest rates: $r_{t} = ln(\frac{p_{t}}{p_{t-1}})\text{ with }p_{t}\text{ = portofolio valuation at timestep }t$. You can customize your own reward function if needed (see https://gym-trading-env.readthedocs.io/en/latest/customization.html#custom-reward-function)

In [ ]:
done, truncated = False, False
observation, info = env_train.reset()
while not done and not truncated:
    # Pick a position by its index in your position list
    position_index = env_train.action_space.sample()
    observation, reward, done, truncated, info = env_train.step(position_index)

### Algorithm and experimental design
DQN is suitable because observations are continuous and the five portfolio positions form a small discrete action space. Replay and a target network stabilize learning. The baseline uses raw features, a 64×64 network, Huber loss, gradient clipping, and identical fees and positions across experiments.

First inspect the baseline training curves, validation drawdown, action changes, and returns relative to cash and buy-and-hold. Small price-return features have very different scales from price ratios and volume: the first controlled change standardizes static features using **fit-period statistics only**. The next change is Double DQN, motivated by potential maximization bias with noisy financial rewards. It changes only the next-action selection rule. Compare three matched seeds and report unsuccessful changes as well as improvements; neither hypothesis guarantees a gain.

The last 30 days of the permitted training year are validation. Choose the configuration by mean validation return across seeds, then retrain on the complete training year with a predeclared seed. The fixed evaluation dates are never used for tuning. Evaluation starts in cash so repeated deterministic episodes are reproducibility checks, not independent market samples.

Environment reference: [Gym Trading Env](https://gym-trading-env.readthedocs.io/en/latest/environment_desc.html).


In [ ]:
class RandomAgent:
    def __init__(self, action_space):
        self.action_space = action_space

    def act(self, observation):
        return self.action_space.sample()

    def choose_action_eval(self, state):
        return self.act(state)


In [ ]:
def evaluate_agent(agent, env, num_episodes=10, max_steps=None, render=False, csv_path="evaluation_results.csv", renderer_logs_dir="render_logs"):
    """
    Evaluate the agent on the environment for a number of episodes.
    """

    results = []
    for ep in range(num_episodes):
        obs, info = env.reset(seed=10000 + ep)
        done = False
        truncated = False
        step = 0
        reward_total = 0.0
        while not done and not truncated:
            action = agent.choose_action_eval(obs)
            obs, reward, done, truncated, info = env.step(action)
            reward_total += reward
            step += 1
            if (max_steps is not None) and (step >= max_steps):
                break

        # Get metrics from the environment
        metrics = env.unwrapped.get_metrics()  
        # Transform metrics
        # Assume metrics contain keys "Portfolio Return" and "Market Return" as strings like "45.24%"
        port_ret = float(metrics["Portfolio Return"].strip('%')) / 100.0
        market_ret = float(metrics["Market Return"].strip('%')) / 100.0

        results.append({
            "episode": ep+1,
            "portfolio_return": port_ret,
            "market_return": market_ret,
            "excess_return": port_ret - market_ret,
            "steps": step,
            "total_reward": reward_total,
        })
        if render:
            print(f"Eval Episode {ep+1}: Total Reward: {reward_total:.2f}, Portfolio Return: {port_ret:.2%}, Market Return: {market_ret:.2%}, Excess Return: {(port_ret - market_ret):.2%}, Steps: {step}")
            time.sleep(1)  # Pause between episodes in case the execution is too fast and files are not saved properly
            env.unwrapped.save_for_render(dir=str(renderer_logs_dir))

    df_results = pd.DataFrame(results)

    df_results.to_csv(csv_path, index=False)
    print(f"Saved submission to {csv_path}")

    return df_results


In [ ]:
import torch
from torch import nn
from collections import deque
import random
from copy import deepcopy

torch.set_num_threads(1)

class TradingDQN:
    """Replay DQN, with optional Double DQN and train-only standardization."""
    def __init__(self, state_size, n_actions, mean, std, double=False, lr=1e-4, seed=23):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        self.n_actions = n_actions
        self.mean = np.asarray(mean, dtype=np.float32)
        self.std = np.asarray(std, dtype=np.float32)
        self.double = double
        self.q_network = nn.Sequential(nn.Linear(state_size, 64), nn.ReLU(),
                                       nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, n_actions))
        self.target = deepcopy(self.q_network).eval()
        self.optimizer = torch.optim.Adam(self.q_network.parameters(), lr=lr)
        self.buffer = deque(maxlen=50000)
        self.steps = 0
        self.epsilon = 1.0
        self.gamma = 0.99

    def encode(self, states):
        return torch.as_tensor((np.asarray(states, dtype=np.float32) - self.mean) / self.std)

    def choose_action_eval(self, state):
        with torch.no_grad():
            return int(self.q_network(self.encode(state)).argmax().item())

    def choose_action(self, state):
        return random.randrange(self.n_actions) if random.random() < self.epsilon else self.choose_action_eval(state)

    def update(self, state, action, reward, next_state, terminated):
        self.buffer.append((state.copy(), action, reward, next_state.copy(), terminated))
        self.steps += 1
        self.epsilon = max(0.05, 1.0 - self.steps / 15000)
        if len(self.buffer) < 256 or self.steps % 4:
            return None
        states, actions, rewards, next_states, terminals = zip(*random.sample(self.buffer, 64))
        states, next_states = self.encode(states), self.encode(next_states)
        actions = torch.tensor(actions, dtype=torch.long).unsqueeze(1)
        current = self.q_network(states).gather(1, actions).squeeze(1)
        with torch.no_grad():
            if self.double:
                selected = self.q_network(next_states).argmax(1, keepdim=True)
                future = self.target(next_states).gather(1, selected).squeeze(1)
            else:
                future = self.target(next_states).max(1).values
            target = torch.tensor(rewards, dtype=torch.float32) + self.gamma * (1 - torch.tensor(terminals, dtype=torch.float32)) * future
        loss = nn.functional.smooth_l1_loss(current, target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_network.parameters(), 10.0)
        self.optimizer.step()
        if self.steps % 1000 == 0:
            self.target.load_state_dict(self.q_network.state_dict())
        return float(loss.item())

    def save(self, path):
        torch.save(dict(weights=self.q_network.state_dict(), mean=self.mean.tolist(), std=self.std.tolist(),
                        double=self.double, state_size=len(self.mean), n_actions=self.n_actions,
                        positions=POSITIONS, epsilon=self.epsilon), path)

    @classmethod
    def load(cls, path):
        data = torch.load(path, map_location='cpu', weights_only=True)
        if data['positions'] != POSITIONS:
            raise ValueError('Checkpoint position mapping differs from this environment')
        agent = cls(data['state_size'], data['n_actions'], data['mean'], data['std'], double=data['double'])
        agent.q_network.load_state_dict(data['weights'])
        agent.target.load_state_dict(data['weights'])
        agent.epsilon = data['epsilon']
        return agent


def make_env(frame, duration='max'):
    return gym.make('TradingEnv', name='BTCUSDT', df=prepare_env_data(frame),
                    positions=POSITIONS, initial_position=0, trading_fees=0.01/100,
                    borrow_interest_rate=0.0003/100, max_episode_duration=duration, verbose=0)


def fit_agent(frame, *, normalize=False, double=False, seed=23, episodes=40):
    columns = [c for c in frame.columns if 'feature' in c]
    # Dynamic position features have a natural scale; do not fit them on validation/test trajectories.
    mean = np.r_[frame[columns].mean().to_numpy(), [0, 0]] if normalize else np.zeros(len(columns)+2)
    std = np.r_[frame[columns].std().clip(lower=1e-6).to_numpy(), [1, 1]] if normalize else np.ones(len(columns)+2)
    agent = TradingDQN(len(mean), len(POSITIONS), mean, std, double=double, seed=seed)
    env = make_env(frame, duration=720)
    logs = []
    try:
        for ep in range(episodes):
            state, _ = env.reset(seed=seed+ep)
            finished = False
            total, losses = 0.0, []
            while not finished:
                action = agent.choose_action(state)
                next_state, reward, terminated, truncated, _ = env.step(action)
                if not np.isfinite(reward):
                    raise FloatingPointError('Non-finite reward; inspect portfolio insolvency')
                loss = agent.update(state, action, reward, next_state, terminated)
                if loss is not None:
                    losses.append(loss)
                total += reward
                state = next_state
                finished = terminated or truncated
            logs.append(dict(episode=ep+1, reward=total, loss=np.mean(losses) if losses else 0, epsilon=agent.epsilon))
    finally:
        env.close()
    return agent, pd.DataFrame(logs)


def score_agent(agent, frame):
    env = make_env(frame)
    state, _ = env.reset(seed=123)
    values = [float(env.unwrapped.historical_info['portfolio_valuation', -1])]
    actions = []
    try:
        finished = False
        while not finished:
            action = agent.choose_action_eval(state)
            state, reward, terminated, truncated, _ = env.step(action)
            values.append(float(env.unwrapped.historical_info['portfolio_valuation', -1]))
            actions.append(action)
            finished = terminated or truncated
    finally:
        env.close()
    values = np.asarray(values)
    market = float(frame.close.iloc[-1]/frame.close.iloc[0]-1)
    return dict(portfolio_return=values[-1]/values[0]-1, market_return=market,
                excess_return=values[-1]/values[0]-1-market,
                max_drawdown=float(np.min(values/np.maximum.accumulate(values)-1)),
                action_changes=int(np.count_nonzero(np.diff(actions))))


class ConstantAgent:
    def __init__(self, position):
        self.action = POSITIONS.index(position)
    def choose_action_eval(self, state):
        return self.action


In [ ]:
# Keep the final month of the allowed training period for model selection.
VALID_START = EVAL_START - pd.Timedelta(days=30)
fit_data = df_train.loc[df_train.index < VALID_START].copy()
valid_data = df_train.loc[df_train.index >= VALID_START].copy()
assert fit_data.index.max() < valid_data.index.min() < EVAL_START
SEEDS = [23, 42, 71]
EPISODES = 40
experiments = [('baseline_dqn', False, False), ('scaled_dqn', True, False), ('scaled_double_dqn', True, True)]
results, training_logs = [], {}
for name, normalize, double in experiments:
    for seed in SEEDS:
        candidate, history = fit_agent(fit_data, normalize=normalize, double=double, seed=seed, episodes=EPISODES)
        metrics = score_agent(candidate, valid_data)
        results.append(dict(experiment=name, seed=seed, **metrics))
        training_logs[(name, seed)] = history
        history.to_csv(eval_folder / f'training_{name}_{seed}.csv', index=False)
        print(name, seed, metrics)
experiment_results = pd.DataFrame(results)
experiment_results.to_csv(eval_folder / 'validation_experiments.csv', index=False)
summary = experiment_results.groupby('experiment').agg(
    mean_return=('portfolio_return', 'mean'), std_return=('portfolio_return', 'std'),
    mean_excess=('excess_return', 'mean'), mean_drawdown=('max_drawdown', 'mean'),
    mean_action_changes=('action_changes', 'mean'))
print(summary)
print('Validation cash:', score_agent(ConstantAgent(0), valid_data))
print('Validation buy and hold:', score_agent(ConstantAgent(1), valid_data))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, _, _ in experiments:
    histories = [training_logs[(name, s)] for s in SEEDS]
    axes[0].plot(np.mean([h.reward for h in histories], axis=0), label=name)
    axes[1].plot(np.mean([h.loss for h in histories], axis=0), label=name)
axes[0].set(title='Mean training log return (three seeds)', xlabel='Episode')
axes[1].set(title='Mean training loss (three seeds)', xlabel='Episode')
for ax in axes:
    ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.show()

baseline = summary.loc['baseline_dqn', 'mean_return']
for name in summary.index:
    delta = summary.loc[name, 'mean_return'] - baseline
    print(f'{name}: validation mean return difference from baseline = {delta:+.2%}; '
          f'seed standard deviation = {summary.loc[name, "std_return"]:.2%}')
selected_name = summary.mean_return.idxmax()
_, normalize, double = next(row for row in experiments if row[0] == selected_name)
print('Selected using validation only:', selected_name)
# Freeze configuration before touching evaluation. Retrain on all allowed training data.
agent, final_history = fit_agent(df_train, normalize=normalize, double=double, seed=23, episodes=EPISODES)
agent.save(eval_folder / 'final_trading_agent.pth')
final_history.to_csv(eval_folder / 'final_training.csv', index=False)
(eval_folder / 'selected_configuration.json').write_text(json.dumps(dict(
    experiment=selected_name, normalize=normalize, double=double, seed=23, episodes=EPISODES,
    validation_start=str(VALID_START), evaluation_start=str(EVAL_START), evaluation_end=str(EVAL_END)), indent=2))


In [ ]:
# The experiment cells above select and train the final agent.
df_results = evaluate_agent(agent, env_eval, num_episodes=10, render=True,
                            csv_path=eval_folder / 'evaluation_results.csv',
                            renderer_logs_dir=eval_folder / 'render_logs')
print(df_results)
print('Final-period cash:', score_agent(ConstantAgent(0), df_eval))
print('Final-period buy and hold:', score_agent(ConstantAgent(1), df_eval))
env_train.close()
env_eval.close()


### Environment rendering

Gym Trading Env supports rendering the environment to visualize the agent's trading actions over time. You can enable rendering during evaluation by setting the `render` parameter to `True` in the `evaluate_agent` function. The rendered logs will be saved in the specified directory for later review.

To visualize the rendered logs, you can use the built-in rendering tools provided by Gym Trading Env as shown below.

In [ ]:
# Opt in to the blocking interactive renderer after the notebook has completed.
RUN_RENDERER = False
if RUN_RENDERER:
    from gym_trading_env.renderer import Renderer
    renderer = Renderer(render_logs_dir=str(eval_folder / 'render_logs'))
    renderer.run()


### Results discussion and presentation outline (10–15 minutes)
1. **Problem and data (2 minutes):** five position choices, causal hourly inputs, costs, chronological split.
2. **Algorithm (3 minutes):** replay, detached targets, epsilon decay, and Double DQN selection/evaluation.
3. **Baseline evidence (2 minutes):** show the saved training curves and validation return, drawdown, and action changes. Discuss whether the policy beats constant positions after costs.
4. **Experiments (3 minutes):** show `validation_experiments.csv` and the summary above; explain normalization and Double DQN, including negative deltas and variability across seeds.
5. **Final evaluation (2 minutes):** show `evaluation_results.csv` and the selected configuration. A deterministic policy on one fixed period gives repeated results; it does not provide ten independent estimates. A validation winner may underperform on the final period because regimes change and selection overfits.

The cells above print measured comparisons, rather than assuming that an optimization worked. Checkpoint, configuration, training histories, and evaluation CSV are saved under `eval/`. Training and evaluation are a historical simulation; conclusions are limited to these data and this experimental budget.
